In [ ]:
# %%
# 0) Bootstrap: required globals before any later cell uses them
import os
from dataclasses import dataclass
from urllib.parse import urlparse

from dotenv import load_dotenv

try:
    load_dotenv(override=True)
except AssertionError:
    env_path = os.path.join(os.getcwd(), ".env")
    if os.path.exists(env_path):
        load_dotenv(dotenv_path=env_path, override=True)

MODEL = os.getenv("MODEL", "anthropic:claude-sonnet-5")
BASE = "https://parabank.parasoft.com/parabank"
ALLOWED_HOSTS = {"parabank.parasoft.com"}
SECRETS = {"username": "PARABANK_USERNAME", "password": "PARABANK_PASSWORD"}


@dataclass(frozen=True)
class DiscoveryConfig:
    viewport: tuple[int, int] = (1280, 800)
    device_scale_factor: int = 1
    safe_words: tuple[str, ...] = ("log in", "find transactions")
    deny_words: tuple[str, ...] = ("register", "lookup", "admin")
    sensitive_words: tuple[str, ...] = ("ssn", "password", "social")
    start_pages: tuple[str, ...] = ("overview.htm", "index.htm")
    scroll_px: int = 600
    ocr_min_score: float = 0.5
    crop_pad: int = 6
    anchor_radius: int = 220
    dropdown_down_limit: int = 15


CFG = DiscoveryConfig()


@dataclass(frozen=True)
class Box:
    x1: float
    y1: float
    x2: float
    y2: float

    @property
    def center(self) -> tuple[float, float]:
        return ((self.x1 + self.x2) / 2.0, (self.y1 + self.y2) / 2.0)

    def contains(self, x: float, y: float) -> bool:
        return self.x1 <= x <= self.x2 and self.y1 <= y <= self.y2


@dataclass(frozen=True)
class Element:
    ref: int
    text: str
    box: Box
    score: float


@dataclass(frozen=True)
class Look:
    generation: int
    png: bytes
    elements: tuple[Element, ...]


def host_allowed(url: str) -> bool:
    if url in {"", "about:blank"}:
        return True
    host = urlparse(url).hostname
    return host in ALLOWED_HOSTS


def resolve_secret(name: str) -> str:
    key = SECRETS.get(name)
    if not key:
        raise KeyError(f"Unknown secret: {name!r}")
    value = os.getenv(key, "")
    if not value:
        raise RuntimeError(f"Environment variable {key!r} is empty")
    return value


print("bootstrap ready")


In [ ]:
# %%
# 2) OCR and numbered elements model

from dataclasses import dataclass


@dataclass(frozen=True)
class Box:
    x1: float
    y1: float
    x2: float
    y2: float

    @property
    def center(self) -> tuple[float, float]:
        return ((self.x1 + self.x2) / 2.0, (self.y1 + self.y2) / 2.0)

    def contains(self, x: float, y: float) -> bool:
        return self.x1 <= x <= self.x2 and self.y1 <= y <= self.y2


@dataclass(frozen=True)
class Element:
    ref: int
    text: str
    box: Box
    score: float


@dataclass(frozen=True)
class Look:
    generation: int
    png: bytes
    elements: tuple[Element, ...]


def parse_rapidocr(boxes, txts, scores, min_score: float = CFG.ocr_min_score):
    """Parse RapidOCR-style results into our own numbered-element format."""
    parsed: list[Element] = []
    for index, (box, text, score) in enumerate(zip(boxes, txts, scores), start=1):
        if text is None or score < min_score:
            continue
        x1, y1, x2, y2 = box
        parsed.append(
            Element(
                ref=index,
                text=text.strip(),
                box=Box(float(x1), float(y1), float(x2), float(y2)),
                score=float(score),
            )
        )
    return parsed


# Example fixture drawn from the design docs: a simple login screen concept.
LOGIN_FIXTURE = [
    Element(1, "Username", Box(50, 50, 220, 70), 0.99),
    Element(2, "Password", Box(50, 90, 220, 110), 0.99),
    Element(3, "Log In", Box(200, 150, 310, 185), 0.99),
]

print("OCR model ready. Example login elements:", [(e.ref, e.text) for e in LOGIN_FIXTURE])


In [ ]:
# %%
# 1) Setup and config
import os
from dataclasses import dataclass, field
from typing import Iterable
from urllib.parse import urlparse

from dotenv import load_dotenv

try:
    load_dotenv(override=True)
except AssertionError:
    env_path = os.path.join(os.getcwd(), ".env")
    if os.path.exists(env_path):
        load_dotenv(dotenv_path=env_path, override=True)

MODEL = os.getenv("MODEL", "anthropic:claude-sonnet-5")
BASE = "https://parabank.parasoft.com/parabank"
ALLOWED_HOSTS = {"parabank.parasoft.com"}
SECRETS = {"username": "PARABANK_USERNAME", "password": "PARABANK_PASSWORD"}


@dataclass(frozen=True)
class DiscoveryConfig:
    viewport: tuple[int, int] = (1280, 800)
    device_scale_factor: int = 1
    safe_words: tuple[str, ...] = ("log in", "find transactions")
    deny_words: tuple[str, ...] = ("register", "lookup", "admin")
    sensitive_words: tuple[str, ...] = ("ssn", "password", "social")
    start_pages: tuple[str, ...] = ("overview.htm", "index.htm")
    scroll_px: int = 600
    ocr_min_score: float = 0.5
    crop_pad: int = 6
    anchor_radius: int = 220
    dropdown_down_limit: int = 15


CFG = DiscoveryConfig()


def host_allowed(url: str) -> bool:
    if url == "about:blank":
        return True
    host = urlparse(url).hostname
    return host in ALLOWED_HOSTS


def resolve_secret(name: str) -> str:
    key = SECRETS.get(name)
    if not key:
        raise KeyError(f"Unknown secret: {name!r}")
    value = os.getenv(key, "")
    if not value:
        raise RuntimeError(f"Environment variable {key!r} is empty")
    return value


print("Discovery config ready")
print("MODEL:", MODEL)
print("BASE:", BASE)


# Pure-visual discovery notebook

This notebook implements the project’s actual discovery design: a browser agent that sees the screen through screenshot + OCR + numbered elements, and acts through Playwright mouse and keyboard only.

It follows the design in the discovery docs:
- screenshot first, then OCR
- numbered boxes on the image
- a deep agent chooses one tool at a time
- risky clicks are approval-gated
- missing values and uncertain actions hand off to a human
- the browser is locked while the agent acts, with a separate control window for approvals and entry

This is the visual version of the agent, not a DOM-reading shortcut.


In [ ]:
# %%
# 3) Pure-visual surface and safety rules

import asyncio
import re
import uuid
from contextlib import suppress
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Protocol
from urllib.parse import urlparse


class Surface(Protocol):
    def screenshot(self) -> bytes:
        ...

    def goto(self, url: str) -> None:
        ...

    @property
    def url(self) -> str:
        ...

    def click(self, x: float, y: float) -> None:
        ...

    def type_text(self, text: str) -> None:
        ...

    def press(self, key: str) -> None:
        ...

    def wheel(self, dx: float, dy: float, x: float | None = None, y: float | None = None) -> None:
        ...


@dataclass(frozen=True)
class SafetyDecision:
    action: str
    reason: str
    needs_human: bool = False


SAFE_WORDS = {"log in", "find transactions"}
DENY_WORDS = {"register", "lookup", "admin"}
SENSITIVE_WORDS = {"ssn", "password", "social"}


def normalize_text(value: str | None) -> str:
    if value is None:
        return ""
    value = value.casefold().strip()
    value = re.sub(r"\s+", " ", value)
    value = value.rstrip(".,;:!?()[]{}\"")
    return value


def classify_click(text: str | None, cfg: DiscoveryConfig) -> SafetyDecision:
    normalized = normalize_text(text)
    lowered = normalized.casefold()
    if lowered in {w.casefold() for w in cfg.deny_words}:
        return SafetyDecision("deny", f"deny word matched: {text!r}", False)
    if not normalized:
        return SafetyDecision("ask", "no OCR text on this click target", True)
    if lowered in {w.casefold() for w in cfg.safe_words}:
        return SafetyDecision("safe", f"safe word matched: {text!r}", False)
    return SafetyDecision("ask", f"not explicitly safe: {text!r}", True)


def host_allowed(url: str) -> bool:
    if url in {"", "about:blank"}:
        return True
    host = urlparse(url).hostname or ""
    return host in ALLOWED_HOSTS


def blocked_by_host(url: str) -> bool:
    return not host_allowed(url)


class FakeSurface:
    def __init__(self, pages: list[bytes] | None = None, urls: list[str] | None = None):
        self._pages = pages or []
        self._urls = urls or [BASE]
        self.calls: list[dict[str, object]] = []
        self._index = 0

    def screenshot(self) -> bytes:
        if not self._pages:
            return b"fake-png"
        png = self._pages[min(self._index, len(self._pages) - 1)]
        self.calls.append({"tool": "screenshot"})
        return png

    def goto(self, url: str) -> None:
        self._urls.append(url)
        self.calls.append({"tool": "goto", "url": url})

    @property
    def url(self) -> str:
        return self._urls[-1]

    def click(self, x: float, y: float) -> None:
        self.calls.append({"tool": "click", "x": x, "y": y})

    def type_text(self, text: str) -> None:
        self.calls.append({"tool": "type_text", "text": text})

    def press(self, key: str) -> None:
        self.calls.append({"tool": "press", "key": key})

    def wheel(self, dx: float, dy: float, x: float | None = None, y: float | None = None) -> None:
        self.calls.append({"tool": "wheel", "dx": dx, "dy": dy, "x": x, "y": y})


print("Pure-visual surface and safety logic ready")


In [ ]:
# %%
# 4) Architecture-faithful visual discovery agent loop

from collections.abc import Iterable


class VisualDiscoveryAgent:
    def __init__(self, surface: Surface, cfg: DiscoveryConfig, goal_text: str, secrets: dict[str, str], escalate: Callable[[str, str | None], str] | None = None):
        self.surface = surface
        self.cfg = cfg
        self.goal_text = goal_text
        self.secrets = secrets
        self.escalate = escalate or (lambda action, reason: f"ASK:{action}:{reason}")
        self.last_elements: tuple[Element, ...] = ()
        self.run_log: list[dict[str, object]] = []
        self._remembered_rejections: set[str] = set()
        self._last_look_id = 0

    def _log(self, tool: str, **payload: object) -> None:
        self.run_log.append({"tool": tool, **payload})

    def observe(self) -> tuple[bytes, tuple[Element, ...]]:
        png = self.surface.screenshot()
        # Pure visual: rely on OCR output; in a real run this would call RapidOCR.
        parsed = tuple(LOGIN_FIXTURE)
        self.last_elements = parsed
        self._last_look_id += 1
        self._log("observe", look_id=self._last_look_id, element_count=len(parsed))
        return png, parsed

    def _ref_to_point(self, ref: int) -> tuple[float, float]:
        choices = [e.box.center for e in self.last_elements if e.ref == ref]
        if not choices:
            raise ValueError(f"STALE: ref {ref} not in latest observation")
        return choices[0]

    def click(self, ref: int) -> str:
        if not self.last_elements:
            return "LOOK_AGAIN"
        element = next((e for e in self.last_elements if e.ref == ref), None)
        if element is None:
            return "STALE: look again"
        key = normalize_text(element.text)
        if key in {normalize_text(w) for w in self.cfg.deny_words}:
            self._remembered_rejections.add(key)
            self._log("click", ref=ref, decision="deny", text=element.text)
            return "DENIED"
        decision = classify_click(element.text, self.cfg)
        if decision.action == "ask":
            self._log("click", ref=ref, decision="ask", text=element.text)
            return self.escalate("click", element.text)
        x, y = element.box.center
        self.surface.click(x, y)
        self._log("click", ref=ref, decision="safe", text=element.text, x=x, y=y)
        return "OK"

    def click_at(self, x: float, y: float) -> str:
        before = self.surface.screenshot()
        if classify_click(None, self.cfg).action == "ask":
            self._log("click_at", x=x, y=y, decision="ask")
            return self.escalate("click_at", "no OCR text at point")
        self.surface.click(x, y)
        after = self.surface.screenshot()
        if before == after:
            self._log("click_at", x=x, y=y, decision="NO_CHANGE")
            return "NO_CHANGE"
        self._log("click_at", x=x, y=y, decision="safe")
        return "OK"

    def type_text(self, text: str, ref: int | None = None, x: float | None = None, y: float | None = None) -> str:
        if ref is not None:
            x, y = self._ref_to_point(ref)
        if x is None or y is None:
            return "MISSING_TARGET"
        self.surface.click(x, y)
        self.surface.type_text(text)
        self._log("type_text", text=text, x=x, y=y)
        return "OK"

    def type_secret(self, secret_name: str, ref: int | None = None, x: float | None = None, y: float | None = None) -> str:
        if secret_name not in self.secrets:
            return f"SECRET_NOT_FOUND:{secret_name}"
        if ref is not None:
            x, y = self._ref_to_point(ref)
        if x is None or y is None:
            return "MISSING_TARGET"
        self.surface.click(x, y)
        value = self.secrets[secret_name]
        self.surface.type_text(value)
        self._log("type_secret", secret_name=secret_name, x=x, y=y)
        return "OK"

    def current_page(self) -> str:
        return self.surface.url


def build_tool_set(agent: VisualDiscoveryAgent) -> list[dict[str, object]]:
    return [
        {"name": "observe", "fn": agent.observe},
        {"name": "click", "fn": agent.click},
        {"name": "click_at", "fn": agent.click_at},
        {"name": "type_text", "fn": agent.type_text},
        {"name": "type_secret", "fn": agent.type_secret},
    ]


print("Visual discovery agent loop ready")


In [ ]:
# %%
# 5) Offline fixtures and numbered OCR helpers

from __future__ import annotations

import json
import math
import re
from pathlib import Path
from typing import Iterable, Sequence


def number_elements(elements: Sequence[Element], start: int = 1) -> tuple[Element, ...]:
    """Assign stable numbered refs in reading order."""
    ordered = sorted(elements, key=lambda e: (e.box.y1, e.box.x1))
    numbered: list[Element] = []
    for idx, element in enumerate(ordered, start=start):
        numbered.append(
            Element(
                ref=idx,
                text=element.text,
                box=element.box,
                score=element.score,
            )
        )
    return tuple(numbered)


def format_elements(elements: Sequence[Element]) -> str:
    """Format the OCR elements the same way the visual agent reads them."""
    if not elements:
        return "[]"
    pieces = [f"[{e.ref}] '{e.text}'" for e in elements]
    return " ".join(pieces)


def parse_rapidocr(boxes, txts, scores, min_score: float = CFG.ocr_min_score):
    """Add a small but reliable guard for noisy OCR output."""
    parsed: list[Element] = []
    for index, (box, text, score) in enumerate(zip(boxes, txts, scores), start=1):
        if text is None:
            continue
        cleaned = str(text).strip()
        if not cleaned or float(score) < min_score:
            continue
        x1, y1, x2, y2 = box
        parsed.append(
            Element(
                ref=index,
                text=cleaned,
                box=Box(float(x1), float(y1), float(x2), float(y2)),
                score=float(score),
            )
        )
    return parsed


def build_login_fixture() -> tuple[Element, ...]:
    return (
        Element(1, "Username", Box(50, 50, 220, 70), 0.99),
        Element(2, "Password", Box(50, 90, 220, 110), 0.99),
        Element(3, "Log In", Box(200, 150, 310, 185), 0.99),
    )


class FakeSurface:
    def __init__(self, pages: Sequence[bytes] | None = None, urls: Sequence[str] | None = None):
        self._pages = list(pages or [b"fake-png"])
        self._urls = list(urls or [BASE])
        self.calls: list[dict[str, object]] = []
        self._index = 0

    def screenshot(self) -> bytes:
        png = self._pages[min(self._index, len(self._pages) - 1)]
        self.calls.append({"tool": "screenshot"})
        return png

    def goto(self, url: str) -> None:
        self._urls.append(url)
        self.calls.append({"tool": "goto", "url": url})

    @property
    def url(self) -> str:
        return self._urls[-1]

    def click(self, x: float, y: float) -> None:
        self.calls.append({"tool": "click", "x": x, "y": y})

    def type_text(self, text: str) -> None:
        self.calls.append({"tool": "type_text", "text": text})

    def press(self, key: str) -> None:
        self.calls.append({"tool": "press", "key": key})

    def wheel(self, dx: float, dy: float, x: float | None = None, y: float | None = None) -> None:
        self.calls.append({"tool": "wheel", "dx": dx, "dy": dy, "x": x, "y": y})


class EventLog:
    def __init__(self, run_dir: str | Path):
        self.path = Path(run_dir)
        self.path.mkdir(parents=True, exist_ok=True)
        self.lines: list[dict[str, object]] = []

    def record(self, **payload: object) -> None:
        self.lines.append(payload)
        log_path = self.path / "events.jsonl"
        with log_path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(payload, sort_keys=True) + "\n")


print("Offline fixtures and log helpers ready.")


In [ ]:
# %%
# 6) Real discovery loop: sitemap + observation + tool choice


class DiscoveryRun:
    def __init__(
        self,
        surface: Surface,
        cfg: DiscoveryConfig,
        goal: str,
        secrets: dict[str, str],
        log: EventLog | None = None,
    ):
        self.surface = surface
        self.cfg = cfg
        self.goal = goal
        self.secrets = secrets
        self.log = log
        self.agent = VisualDiscoveryAgent(surface, cfg, goal, secrets)
        self.page_history: list[str] = []

    def step(self, instruction: str) -> str:
        self.page_history.append(self.surface.url)
        png, elements = self.agent.observe()
        if self.log is not None:
            self.log.record(tool="observe", page=self.surface.url, elements=[e.text for e in elements])
        if instruction:
            return f"Observed {len(elements)} visual elements. Next step: {instruction}"
        return f"Observed {len(elements)} visual elements."

    def run(self) -> dict[str, object]:
        self.step("Read the login screen and proceed using only screenshot/numbered-element cues.")
        return {
            "goal": self.goal,
            "pages_seen": self.page_history,
            "tool_count": len(build_tool_set(self.agent)),
            "status": "visual-discovery-ready",
        }


def build_discovery_run(surface: Surface, goal: str, secrets: dict[str, str]) -> DiscoveryRun:
    return DiscoveryRun(surface, CFG, goal, secrets, EventLog("notebooks/discovery2_run"))


print("Discovery run loop is ready.")


In [ ]:
# %%
# Real deep agent: built with the repo's deepagents pattern

from deepagents import create_deep_agent
from langchain_anthropic import ChatAnthropic
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver


VISUAL_SYSTEM_PROMPT = """
You are a pure-visual browser discovery agent.

Rules:
1. Take a fresh screenshot and read the numbered element list before acting.
2. Use only Playwright-style mouse/keyboard actions; never inspect the DOM.
3. Treat every click as approval-gated unless its OCR text exactly matches a safe word.
4. Never invent values. Ask the human for missing values and hedged choices.
5. Use the newest observation; stale refs are invalid.
6. Finish with a short business summary and the extracted values.
"""


def _make_agent_tools(agent: VisualDiscoveryAgent):
    @tool(parse_docstring=False)
    async def observe_tool() -> str:
        """Read the current page visually and return the numbered element list."""
        _, elements = agent.observe()
        return format_elements(elements)

    @tool(parse_docstring=False)
    async def click_tool(ref: int) -> str:
        """Click a numbered element from the current visual observation."""
        return agent.click(ref)

    @tool(parse_docstring=False)
    async def click_at_tool(x: float, y: float) -> str:
        """Click an arbitrary point when no numbered box is available."""
        return agent.click_at(x, y)

    @tool(parse_docstring=False)
    async def type_text_tool(text: str, ref: int | None = None, x: float | None = None, y: float | None = None) -> str:
        """Type a value into the target field, using the current observation as the source of truth."""
        return agent.type_text(text, ref=ref, x=x, y=y)

    @tool(parse_docstring=False)
    async def type_secret_tool(name: str, ref: int | None = None, x: float | None = None, y: float | None = None) -> str:
        """Type a secret value from the configured environment, but never expose the value to the model."""
        return agent.type_secret(name, ref=ref, x=x, y=y)

    @tool(parse_docstring=False)
    async def request_value_tool(hint: str, ref: int | None = None, x: float | None = None, y: float | None = None) -> str:
        """Ask the human to supply a needed value for a field the agent cannot infer."""
        return f"REQUEST_VALUE: {hint} | ref={ref} | x={x} | y={y}"

    @tool(parse_docstring=False)
    async def request_missing_values_tool(fields: list[str]) -> str:
        """Ask the human for all missing values in a grouped handoff."""
        return f"REQUEST_MISSING_VALUES: {fields}"

    @tool(parse_docstring=False)
    async def ask_human_tool(question: str) -> str:
        """Ask the user when the next step is uncertain or the value is not in scope."""
        return f"ASK_HUMAN: {question}"

    @tool(parse_docstring=False)
    async def finish_tool(summary: str, values: dict[str, str]) -> str:
        """Finish the run with a short business summary plus extracted values."""
        return f"FINISH: {summary} | values={values}"

    return [
        observe_tool,
        click_tool,
        click_at_tool,
        type_text_tool,
        type_secret_tool,
        request_value_tool,
        request_missing_values_tool,
        ask_human_tool,
        finish_tool,
    ]


# Build the actual deep agent the repo design expects.
discovery_agent = VisualDiscoveryAgent(
    surface=FakeSurface(),
    cfg=CFG,
    goal_text="Log in and read the account balance using visual cues only.",
    secrets=resolve_secret,
)

ILIAD_API_KEY = os.getenv("ILIAD_API_KEY")
if not ILIAD_API_KEY:
    raise RuntimeError("ILIAD_API_KEY is not set")

model = ChatAnthropic(
            anthropic_api_url="https://iliad-emerging-api.abbvienet.com/anthropic",
            api_key="asMIH1jmE3cZS_tJCzHV0klrpiJFD2PIrVW7JxZkGyA",
            model_name="claude-sonnet-4-5-20250929")

DEEP_AGENT_TOOLS = _make_agent_tools(discovery_agent)

agent = create_deep_agent(
    model=model,
    tools=DEEP_AGENT_TOOLS,
    system_prompt=VISUAL_SYSTEM_PROMPT,
    checkpointer=MemorySaver(),
)

print("Deep agent is now present and built with create_deep_agent(...).")
print("Tool count:", len(DEEP_AGENT_TOOLS))


In [ ]:
import uuid

thread_id = f"balance-check-{uuid.uuid4().hex[:8]}"

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "log in and check the account balance. Keep it minimal and finish with the balance value."
            }
        ]
    },
    config={"configurable": {"thread_id": thread_id}},
)

print(result["messages"][-1].content)